In [64]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pickle

# from sklearn.datasets import fetch_california_housing, load_digits
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, FunctionTransformer

from sklearn.linear_model import LogisticRegression, Lasso, Ridge

from sklearn.model_selection import train_test_split

from sklearn.metrics import mean_absolute_error, r2_score, classification_report, roc_curve

In [9]:
diabetic_data = pd.read_csv("data/diabetic_data.csv")

In [10]:
diabetic_data.columns

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='object')

In [11]:
diabetic_attr = diabetic_data.drop(columns="readmitted")
diabetic_target = diabetic_data.readmitted

In [12]:
diabetic_attr_dummies = pd.get_dummies(diabetic_attr)

In [13]:
scaler = MinMaxScaler()

In [14]:
diabetic_attr_scaled = scaler.fit_transform(diabetic_attr_dummies)

In [15]:
log_regression = LogisticRegression()

In [16]:
log_regression.fit(diabetic_attr_scaled, diabetic_data.readmitted)

c:\Users\Svetoslav\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [17]:
sample_data = diabetic_data.sample(5000, random_state=42)
sample_attr = sample_data.drop(columns="readmitted")
sample_target = sample_data.readmitted

In [18]:
# GEt all string columns
categorial_columns = sample_attr.dtypes[sample_data.dtypes == np.object_].index.values
categorial_columns

array(['race', 'gender', 'age', 'weight', 'payer_code',
       'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum',
       'A1Cresult', 'metformin', 'repaglinide', 'nateglinide',
       'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide',
       'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone',
       'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide',
       'citoglipton', 'insulin', 'glyburide-metformin',
       'glipizide-metformin', 'glimepiride-pioglitazone',
       'metformin-rosiglitazone', 'metformin-pioglitazone', 'change',
       'diabetesMed'], dtype=object)

In [19]:
numerical_columns = [
    'admission_type_id', 'discharge_disposition_id',
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses']

In [20]:
number_processor = Pipeline([
    ("log_transformer", FunctionTransformer(lambda x: np.log10(x+1e-10))),
    ("minmax", MinMaxScaler())
])

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder() , categorial_columns),
    ("numerical", number_processor, numerical_columns)
])

In [21]:
preprocessor

ColumnTransformer(transformers=[('categorical', OneHotEncoder(),
                                 array(['race', 'gender', 'age', 'weight', 'payer_code',
       'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum',
       'A1Cresult', 'metformin', 'repaglinide', 'nateglinide',
       'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide',
       'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone',
       'acar...
       'diabetesMed'], dtype=object)),
                                ('numerical',
                                 Pipeline(steps=[('log_transformer',
                                                  FunctionTransformer(func=<function <lambda> at 0x0000013E60CCF740>)),
                                                 ('minmax', MinMaxScaler())]),
                                 ['admission_type_id',
                                  'discharge_disposition_id',
                                  'time_in_hospital', 'num_lab_procedures',
                                  'num_procedures', 'num_medications',
                                  'number_outpatient', 'number_emergency',
                                  'number_inpatient', 'number_diagnoses'])])

In [22]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression())
])

In [23]:
pipeline.fit(sample_attr, sample_target)

c:\Users\Svetoslav\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(),
                                                  array(['race', 'gender', 'age', 'weight', 'payer_code',
       'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum',
       'A1Cresult', 'metformin', 'repaglinide', 'nateglinide',
       'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide',
       'glyburide', 'tolbutamide', 'pio...
                                                  Pipeline(steps=[('log_transformer',
                                                                   FunctionTransformer(func=<function <lambda> at 0x0000013E60CCF740>)),
                                                                  ('minmax',
                                                                   MinMaxScaler())]),
                                                  ['admission_type_id',
                                                   'discharge_disposition_id',
                                                   'time_in_hospital',
                                                   'num_lab_procedures',
                                                   'num_procedures',
                                                   'num_medications',
                                                   'number_outpatient',
                                                   'number_emergency',
                                                   'number_inpatient',
                                                   'number_diagnoses'])])),
                ('classifier', LogisticRegression())])

In [24]:
pipeline.score(sample_attr, sample_target)

0.6512

In [27]:
pipeline["classifier"].coef_

array([[-0.55656328,  0.02254068,  0.04394579, ...,  0.13460505,
         0.3924086 , -0.13094903],
       [ 0.13007875,  0.08803615,  0.03773322, ...,  0.1953232 ,
         0.1794212 ,  0.59111607],
       [ 0.42648454, -0.11057683, -0.08167902, ..., -0.32992825,
        -0.5718298 , -0.46016703]])

In [29]:
def getPipeline(c):
    return Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(C=c))
])

In [30]:
p1 = getPipeline(0.00001)
p1.fit(sample_attr, sample_target)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(),
                                                  array(['race', 'gender', 'age', 'weight', 'payer_code',
       'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum',
       'A1Cresult', 'metformin', 'repaglinide', 'nateglinide',
       'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide',
       'glyburide', 'tolbutamide', 'pio...
                                                  Pipeline(steps=[('log_transformer',
                                                                   FunctionTransformer(func=<function <lambda> at 0x0000013E60CCF740>)),
                                                                  ('minmax',
                                                                   MinMaxScaler())]),
                                                  ['admission_type_id',
                                                   'discharge_disposition_id',
                                                   'time_in_hospital',
                                                   'num_lab_procedures',
                                                   'num_procedures',
                                                   'num_medications',
                                                   'number_outpatient',
                                                   'number_emergency',
                                                   'number_inpatient',
                                                   'number_diagnoses'])])),
                ('classifier', LogisticRegression(C=1e-05))])

In [31]:
p1["classifier"].coef_

array([[-7.87262909e-05,  3.04631236e-05, -7.32181705e-06, ...,
         1.79039710e-04,  7.65733413e-04,  1.10366618e-04],
       [-1.36870361e-04, -6.16579738e-05, -2.41907918e-05, ...,
         6.62020400e-04,  1.51604950e-03,  3.14129518e-04],
       [ 2.15596652e-04,  3.11948503e-05,  3.15126088e-05, ...,
        -8.41060110e-04, -2.28178292e-03, -4.24496136e-04]])

In [45]:
(attr_train, attr_test, target_train, target_test ) = train_test_split(diabetic_attr, diabetic_target, test_size= 10*1000, stratify=diabetic_target)

In [44]:
attr_train.shape, attr_test.shape, target_train.shape, target_test.shape

((91766, 49), (10000, 49), (91766,), (10000,))

In [46]:
diabetic_target.value_counts(normalize=True)

readmitted
NO     0.539119
>30    0.349282
<30    0.111599
Name: proportion, dtype: float64

In [47]:
target_train.value_counts(normalize=True)


readmitted
NO     0.539121
>30    0.349280
<30    0.111599
Name: proportion, dtype: float64

In [48]:
target_test.value_counts(normalize=True)


readmitted
NO     0.5391
>30    0.3493
<30    0.1116
Name: proportion, dtype: float64

In [56]:
pipeline1 = getPipeline(100000)

In [57]:
pipeline1.fit(attr_train, target_train)

c:\Users\Svetoslav\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(),
                                                  array(['race', 'gender', 'age', 'weight', 'payer_code',
       'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum',
       'A1Cresult', 'metformin', 'repaglinide', 'nateglinide',
       'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide',
       'glyburide', 'tolbutamide', 'pio...
                                                  Pipeline(steps=[('log_transformer',
                                                                   FunctionTransformer(func=<function <lambda> at 0x0000013E60CCF740>)),
                                                                  ('minmax',
                                                                   MinMaxScaler())]),
                                                  ['admission_type_id',
                                                   'discharge_disposition_id',
                                                   'time_in_hospital',
                                                   'num_lab_procedures',
                                                   'num_procedures',
                                                   'num_medications',
                                                   'number_outpatient',
                                                   'number_emergency',
                                                   'number_inpatient',
                                                   'number_diagnoses'])])),
                ('classifier', LogisticRegression(C=100000))])

In [58]:
pipeline1.score(attr_train, target_train)

0.5875160734912713

In [62]:
# pipeline1.score(attr_test, target_test)
print(classification_report(target_train, pipeline1.predict(attr_train)))

              precision    recall  f1-score   support

         <30       0.37      0.01      0.02     10241
         >30       0.51      0.39      0.44     32052
          NO       0.62      0.83      0.71     49473

    accuracy                           0.59     91766
   macro avg       0.50      0.41      0.39     91766
weighted avg       0.55      0.59      0.54     91766



In [66]:
## use wiht decision_function OR predict_proba, but never with predict
roc_curve(pipeline1.predict_proba(attr_train), target_train)

ValueError: multiclass format is not supported